## Attention Mechanism


### Dot Product

In [12]:
import torch 
import torch.nn as nn
torch.manual_seed(123)
inputs = torch.rand(4,8) #(num_tokens,embedding_dim)
print(inputs.shape)

torch.Size([4, 8])


In [13]:
attn_scores = inputs @ inputs.T
print(attn_scores)
print(attn_scores.shape)

tensor([[1.6775, 1.2426, 1.7782, 1.3842],
        [1.2426, 1.4390, 1.9746, 1.8711],
        [1.7782, 1.9746, 3.7323, 3.3564],
        [1.3842, 1.8711, 3.3564, 3.5359]])
torch.Size([4, 4])


In [14]:
attn_weights = torch.softmax(attn_scores, dim=-1) 
print(attn_weights)

tensor([[0.2858, 0.1850, 0.3161, 0.2131],
        [0.1620, 0.1972, 0.3369, 0.3038],
        [0.0708, 0.0862, 0.4998, 0.3432],
        [0.0543, 0.0884, 0.3903, 0.4670]])


### Self Attention


In [4]:
class SelfAttention(nn.Module):
    def __init__(self, d_in,d_out,qkv_bias):
        super().__init__()
        self.w_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,x):
        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)
    
        attn_scores = queries @  keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vector = attn_weights @ values
        return context_vector

In [5]:
d_in = 8
d_out = 6   
x = torch.rand(10,8)
attn = SelfAttention(d_in,d_out,False)
context_vector = attn(x)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

tensor([[ 0.1229,  0.1945, -0.0422, -0.2549, -0.1012,  0.1217],
        [ 0.1220,  0.1912, -0.0429, -0.2546, -0.1003,  0.1217],
        [ 0.1219,  0.2023, -0.0409, -0.2562, -0.1030,  0.1216],
        [ 0.1233,  0.1945, -0.0429, -0.2541, -0.1006,  0.1212],
        [ 0.1223,  0.1882, -0.0442, -0.2528, -0.0982,  0.1213],
        [ 0.1224,  0.1910, -0.0428, -0.2544, -0.1000,  0.1220],
        [ 0.1218,  0.2004, -0.0428, -0.2532, -0.0993,  0.1202],
        [ 0.1225,  0.1932, -0.0430, -0.2538, -0.0996,  0.1213],
        [ 0.1225,  0.1927, -0.0422, -0.2552, -0.1012,  0.1221],
        [ 0.1217,  0.2007, -0.0419, -0.2551, -0.1016,  0.1209]],
       grad_fn=<MmBackward0>)
Context Vector Shape: torch.Size([10, 6])


### Causal Attention

In [6]:
import torch
import torch.nn as nn
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,droput,qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.w_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.droput = nn.Dropout(droput)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens , d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights = torch.softmax(attn_scores/queries.shape[-1]**0.5,dim=-1)
        attn_weights = self.droput(attn_weights)

        context_vector = attn_weights @ values
        return context_vector
    

In [7]:
torch.manual_seed(123)

input = torch.rand(10,8)
batch = torch.stack((input,input),dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
ca = CausalAttention(d_in,d_out,context_length,0.0)
context_vector = ca(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")


torch.Size([2, 10, 8])
tensor([[[-0.0760, -0.5336,  0.1025, -0.1544,  0.0059, -0.2706],
         [-0.0761, -0.5327,  0.0994, -0.1532,  0.0049, -0.2711],
         [-0.0766, -0.5345,  0.0971, -0.1516,  0.0038, -0.2719],
         [-0.0765, -0.5339,  0.0944, -0.1502,  0.0024, -0.2707],
         [-0.0760, -0.5310,  0.0978, -0.1526,  0.0047, -0.2708],
         [-0.0765, -0.5338,  0.0997, -0.1530,  0.0049, -0.2712],
         [-0.0757, -0.5324,  0.0955, -0.1505,  0.0027, -0.2679],
         [-0.0755, -0.5330,  0.0963, -0.1512,  0.0032, -0.2694],
         [-0.0768, -0.5339,  0.1021, -0.1549,  0.0064, -0.2741],
         [-0.0763, -0.5327,  0.0941, -0.1507,  0.0028, -0.2720]],

        [[-0.0760, -0.5336,  0.1025, -0.1544,  0.0059, -0.2706],
         [-0.0761, -0.5327,  0.0994, -0.1532,  0.0049, -0.2711],
         [-0.0766, -0.5345,  0.0971, -0.1516,  0.0038, -0.2719],
         [-0.0765, -0.5339,  0.0944, -0.1502,  0.0024, -0.2707],
         [-0.0760, -0.5310,  0.0978, -0.1526,  0.0047, -0.2708],


### Multi Head Attention

In [8]:
import torch
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,droput,num_heads,qkv_bias = False):
        super().__init__()
        assert (d_out%num_heads==0) , "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out//num_heads

        self.w_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(droput)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        #(b,num_tokens,d_in) -> (b,num_tokens,num_heads,head_dim)
        queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
        keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values = values.view(b,num_tokens,self.num_heads,self.head_dim)

        # Transpose: (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim)
        queries = queries.transpose(1,2)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3) 

        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]

        attn_scores.masked_fill(mask_bool,-torch.inf)

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_scores @ values).transpose(1,2)
        
        context_vector = context_vector.contiguous().view(b,num_tokens,self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector
        

In [9]:
import torch
torch.manual_seed(123)

inputs = torch.rand(10,8)
batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

batch_size, context_length , d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)
context_vector = mha(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

torch.Size([2, 10, 8])
tensor([[[-0.0348,  0.0913, -0.3014,  0.4339,  0.3169,  0.1672],
         [-0.2907,  0.1028, -0.2143,  0.3709,  0.4754,  0.1350],
         [-0.2558,  0.1046, -0.2306,  0.4212,  0.4138,  0.1120],
         [-0.4640,  0.1197, -0.1597,  0.3605,  0.5647,  0.0796],
         [-0.6494,  0.0976, -0.0992,  0.2702,  0.6901,  0.1221],
         [-0.1541,  0.1457, -0.2584,  0.4040,  0.5015,  0.0889],
         [-0.7326,  0.0513, -0.0841,  0.2842,  0.5999,  0.1489],
         [-0.5212, -0.0450, -0.1591,  0.3875,  0.1911,  0.2648],
         [ 0.0269,  0.2226, -0.2960,  0.4035,  0.6054,  0.0342],
         [-0.5168,  0.0840, -0.1404,  0.3537,  0.4926,  0.1226]],

        [[-0.0348,  0.0913, -0.3014,  0.4339,  0.3169,  0.1672],
         [-0.2907,  0.1028, -0.2143,  0.3709,  0.4754,  0.1350],
         [-0.2558,  0.1046, -0.2306,  0.4212,  0.4138,  0.1120],
         [-0.4640,  0.1197, -0.1597,  0.3605,  0.5647,  0.0796],
         [-0.6494,  0.0976, -0.0992,  0.2702,  0.6901,  0.1221],


### KV Cache

### Multi Query Attention

### Grouped Query Attention

### Multi Head Latent Attention